<a href="https://colab.research.google.com/github/RashiBista/ASLL/blob/main/Nepali_NMT_Transformer_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nepali Neural Machine Translation with Transformers
### English &harr; नेपाली | Transformer-based NMT

**Nepal College of Information Technology (NCIT)** &middot; AI / Machine Learning

---

This notebook teaches **machine translation with Transformer models** in three progressive parts:

| Part | What you build | Effort |
|------|----------------|--------|
| **A** | Instant translation using a **pretrained** transformer (Meta's **NLLB-200**) | Runs in ~2 min |
| **B** | A second pretrained model (**mBART-50**) + **BLEU** evaluation | ~3 min |
| **C** | A **Transformer built from scratch** in PyTorch, trained on an English&ndash;Nepali corpus | ~5 min train |
| **D** | An interactive **Gradio** translation app | optional |

> **How to run:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
> A GPU is recommended but the notebook also runs on CPU (slower).

**Nepali language codes** used across models:
- NLLB: `npi_Deva` (Nepali, Devanagari script) &middot; English: `eng_Latn`
- mBART-50: `ne_NP` &middot; English: `en_XX`


## 0. Setup &mdash; install libraries & check the GPU

In [ ]:
# Install the required libraries (quiet). Takes ~1-2 minutes on first run.
!pip install -q -U transformers sentencepiece sacrebleu sacremoses datasets gradio

import torch
print("PyTorch      :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
print("Using device :", device)

## 1. How a Transformer translates (the intuition)

A translation Transformer is an **encoder&ndash;decoder** network. Given a source sentence it produces a target sentence, one token at a time.

```
   English tokens                              Nepali tokens
   [The, weather, is, nice]                    [<bos>, आज, मौसम, राम्रो, छ, <eos>]
          |                                              ^
          v                                              |
   +----------------+   memory (context)   +----------------------+
   |    ENCODER     | -------------------> |       DECODER        |
   | self-attention |                      | masked self-attention|
   | x N layers     |                      | + cross-attention    |
   +----------------+                      | x N layers           |
                                           +----------------------+
```

Three ideas do all the work:

1. **Self-attention** &mdash; every word looks at every other word in *its own* sentence to build a context-aware representation. This is how the model knows that "bank" near "river" differs from "bank" near "money".
2. **Cross-attention** &mdash; while generating each Nepali word, the decoder attends back to the *encoded English* sentence, choosing which source words matter right now.
3. **Positional encoding** &mdash; attention has no built-in sense of order, so we inject position information into the embeddings (a sentence is not a bag of words).

The decoder uses **masked** self-attention so that when predicting word *t* it can only see words *1...t-1* &mdash; it cannot cheat by looking at the future.

Modern multilingual models (NLLB, mBART, mT5) are exactly this architecture, pretrained on hundreds of languages including Nepali. We start by *using* one, then *build* one.

## Part A &mdash; Instant translation with a pretrained Transformer (NLLB-200)

**NLLB-200** ("No Language Left Behind", Meta AI) is a Transformer trained on 200 languages, including Nepali. We use the distilled 600M-parameter version so it fits comfortably in Colab.

> **Why not `pipeline("translation", ...)`?** In **transformers v5** the `translation` (and `text2text-generation`, `summarization`) pipeline tasks were **removed** &mdash; calling them raises `KeyError: "Unknown task translation"`. We instead call the model directly with `forced_bos_token_id`, which is the approach in Hugging Face's own NLLB docs and works on **both v4 and v5**. It also lets us control beam search, which the pipeline never exposed.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

NLLB_MODEL = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL).to(device)

def nllb_translate(texts, src_lang, tgt_lang, num_beams=4, max_length=128):
    # Batch-translate a string or list of strings with NLLB. Returns a list of strings.
    if isinstance(texts, str):
        texts = [texts]
    tokenizer.src_lang = src_lang                       # set the source language
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=max_length).to(device)
    # convert_tokens_to_ids is version-safe; the old tokenizer.lang_code_to_id was removed
    tgt_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    gen = model.generate(**enc, forced_bos_token_id=tgt_id,
                         num_beams=num_beams, max_length=max_length)
    return tokenizer.batch_decode(gen, skip_special_tokens=True)

# Thin wrappers that mimic the old pipeline output shape: [{"translation_text": ...}]
def en2ne(text, max_length=128):
    return [{"translation_text": t} for t in nllb_translate(text, "eng_Latn", "npi_Deva", max_length=max_length)]
def ne2en(text, max_length=128):
    return [{"translation_text": t} for t in nllb_translate(text, "npi_Deva", "eng_Latn", max_length=max_length)]

print("NLLB translators ready (works on transformers v4 and v5).")

In [ ]:
# Try English -> Nepali
english_sentences = [
    "Kathmandu is the capital of Nepal.",
    "I am a computer engineering student at NCIT.",
    "The weather in Pokhara is beautiful today.",
    "Machine learning is transforming the world.",
    "Please send me the assignment before Friday.",
]

for s in english_sentences:
    out = en2ne(s, max_length=128)[0]["translation_text"]
    print(f"EN: {s}\nNE: {out}\n")

In [ ]:
# Try Nepali -> English
nepali_sentences = [
    "नमस्ते, तपाईंलाई कस्तो छ?",
    "म काठमाडौंमा बस्छु।",
    "आज मौसम धेरै राम्रो छ।",
    "मलाई नेपाली खाना मन पर्छ।",
    "यो परियोजना अर्को हप्ता बुझाउनु पर्छ।",
]

for s in nepali_sentences:
    out = ne2en(s, max_length=128)[0]["translation_text"]
    print(f"NE: {s}\nEN: {out}\n")

### A.1 &mdash; Batch translation & what's happening inside

`nllb_translate` (defined above) is the reusable workhorse. Two details make NLLB work: we set `tokenizer.src_lang` so the encoder knows the input language, and we pass `forced_bos_token_id` so the decoder's **first** generated token is the target-language tag &mdash; that single forced token is what selects Nepali vs. English output. Batching many sentences in one call is far faster than looping.

In [ ]:
# Batch example (efficient: all sentences translated in one call)
batch = [
    "Nepal has eight of the ten highest mountains in the world.",
    "The students are working on a natural language processing project.",
    "eSewa and Khalti are popular digital wallets in Nepal.",
]
for src, tr in zip(batch, nllb_translate(batch, "eng_Latn", "npi_Deva")):
    print(f"EN: {src}\nNE: {tr}\n")

## Part B &mdash; A second model (mBART-50) and how to *measure* quality

Different pretrained transformers give different translations. **mBART-50** (Meta) is another many-to-many model covering Nepali (`ne_NP`). Comparing models is good practice.

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

MBART_MODEL = "facebook/mbart-large-50-many-to-many-mmt"
mbart_tok = MBart50TokenizerFast.from_pretrained(MBART_MODEL)
mbart = MBartForConditionalGeneration.from_pretrained(MBART_MODEL).to(device)

def mbart_translate(texts, src_lang, tgt_lang, num_beams=4, max_length=128):
    if isinstance(texts, str):
        texts = [texts]
    mbart_tok.src_lang = src_lang
    enc = mbart_tok(texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=max_length).to(device)
    # convert_tokens_to_ids is version-safe (mbart_tok.lang_code_to_id was removed in v5)
    tgt_id = mbart_tok.convert_tokens_to_ids(tgt_lang)
    gen = mbart.generate(**enc, forced_bos_token_id=tgt_id,
                         num_beams=num_beams, max_length=max_length)
    return mbart_tok.batch_decode(gen, skip_special_tokens=True)

print("mBART-50 ready.")

In [ ]:
# Side-by-side comparison: NLLB vs mBART-50 (English -> Nepali)
compare = [
    "Artificial intelligence will change education in Nepal.",
    "Could you please help me solve this programming problem?",
]
nllb_out  = nllb_translate(compare, "eng_Latn", "npi_Deva")
mbart_out = mbart_translate(compare, "en_XX", "ne_NP")

for src, a, b in zip(compare, nllb_out, mbart_out):
    print(f"EN     : {src}")
    print(f"NLLB   : {a}")
    print(f"mBART50: {b}\n")

### B.1 &mdash; BLEU: a numeric score for translation quality

**BLEU** compares the machine translation against one or more human **reference** translations by measuring n-gram overlap (0&ndash;100; higher is better). We use `sacreBLEU`, the standard implementation.

Below is a tiny English&ndash;Nepali test set with human references. In a real evaluation you would use a held-out benchmark such as **FLORES-200**.

In [ ]:
import sacrebleu

# Small hand-checked test set: (English source, human Nepali reference)
test_set = [
    ("Good morning, teacher.",              "शुभ प्रभात, गुरुजी।"),
    ("I study at Nepal College of Information Technology.",
                                            "म नेपाल कलेज अफ इन्फर्मेसन टेक्नोलोजीमा पढ्छु।"),
    ("Today is a public holiday.",          "आज सार्वजनिक बिदा हो।"),
    ("Water is essential for life.",        "पानी जीवनको लागि आवश्यक छ।"),
    ("The bus to Pokhara leaves at seven.", "पोखरा जाने बस सात बजे छुट्छ।"),
]
sources    = [s for s, _ in test_set]
references = [r for _, r in test_set]

for name, fn, sl, tl in [("NLLB",  nllb_translate, "eng_Latn", "npi_Deva"),
                         ("mBART", mbart_translate, "en_XX",    "ne_NP")]:
    hyps = fn(sources, sl, tl)
    # sacreBLEU expects references as a list-of-lists (one list per reference set)
    bleu = sacrebleu.corpus_bleu(hyps, [references], tokenize="flores200"
                                 if name == "NLLB" else "13a")
    print(f"{name:6s} BLEU = {bleu.score:.2f}")
    for src, hyp, ref in zip(sources, hyps, references):
        print(f"   src: {src}\n   hyp: {hyp}\n   ref: {ref}\n")

## Part C &mdash; Build a Transformer from scratch (PyTorch)

Now we implement the encoder&ndash;decoder Transformer ourselves and train it on a small English&ndash;Nepali parallel corpus. The goal is **understanding the mechanics**, not state-of-the-art quality &mdash; with only a few dozen sentences the model essentially memorises the mapping, which is exactly what we want for a teaching demo.

Steps: (1) a parallel corpus &rarr; (2) tokenisation & vocab &rarr; (3) the model &rarr; (4) training &rarr; (5) greedy decoding.

### C.1 &mdash; A small parallel corpus (Nepal-contextualised)

In [ ]:
# English -> Nepali sentence pairs. Word-level (space-separated) for simplicity.
pairs = [
    ("hello",                         "नमस्ते"),
    ("thank you",                     "धन्यवाद"),
    ("good morning",                  "शुभ प्रभात"),
    ("good night",                    "शुभ रात्री"),
    ("how are you",                   "तपाईंलाई कस्तो छ"),
    ("i am fine",                     "म ठिक छु"),
    ("what is your name",             "तपाईंको नाम के हो"),
    ("my name is ram",                "मेरो नाम राम हो"),
    ("i am a student",                "म विद्यार्थी हुँ"),
    ("i am a teacher",                "म शिक्षक हुँ"),
    ("i love nepal",                  "म नेपाललाई माया गर्छु"),
    ("nepal is beautiful",            "नेपाल सुन्दर छ"),
    ("kathmandu is the capital",      "काठमाडौं राजधानी हो"),
    ("the weather is nice",           "मौसम राम्रो छ"),
    ("it is raining today",           "आज पानी परिरहेको छ"),
    ("where do you live",             "तपाईं कहाँ बस्नुहुन्छ"),
    ("i live in kathmandu",           "म काठमाडौंमा बस्छु"),
    ("i like nepali food",            "मलाई नेपाली खाना मन पर्छ"),
    ("water is life",                 "पानी जीवन हो"),
    ("this is my book",               "यो मेरो किताब हो"),
    ("that is a mountain",            "त्यो पहाड हो"),
    ("the sun is rising",             "सूर्य उदाइरहेको छ"),
    ("i am going home",               "म घर जाँदैछु"),
    ("please help me",                "कृपया मलाई मद्दत गर्नुहोस्"),
    ("see you tomorrow",              "भोलि भेटौंला"),
    ("i am learning nepali",          "म नेपाली सिक्दैछु"),
    ("the class starts now",          "कक्षा अहिले सुरु हुन्छ"),
    ("he is my friend",               "उनी मेरो साथी हुन्"),
    ("she is a doctor",               "उनी डाक्टर हुन्"),
    ("we are happy",                  "हामी खुसी छौं"),
    ("the food is delicious",         "खाना मिठो छ"),
    ("i have a computer",             "मसँग कम्प्युटर छ"),
    ("the book is on the table",      "किताब टेबलमा छ"),
    ("i drink tea every morning",     "म हरेक बिहान चिया पिउँछु"),
    ("the children are playing",      "बच्चाहरू खेलिरहेका छन्"),
    ("today is a holiday",            "आज बिदा हो"),
    ("i am very tired",               "म धेरै थकित छु"),
    ("come here please",              "कृपया यहाँ आउनुहोस्"),
    ("the market is far",             "बजार टाढा छ"),
    ("i will call you later",         "म तपाईंलाई पछि फोन गर्छु"),
]
print(f"{len(pairs)} sentence pairs loaded.")
print("Example ->", pairs[10])

### C.2 &mdash; Tokenisation, vocabulary and tensors

We build a word-level vocabulary for each language with four special tokens: `<pad>`, `<bos>` (begin), `<eos>` (end), `<unk>` (unknown).

In [ ]:
from collections import Counter

PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"
SPECIALS = [PAD, BOS, EOS, UNK]
PAD_IDX, BOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

def tokenize(text):
    return text.strip().split()

def build_vocab(sentences):
    counter = Counter()
    for s in sentences:
        counter.update(tokenize(s))
    itos = SPECIALS + sorted(counter)          # index -> string
    stoi = {tok: i for i, tok in enumerate(itos)}  # string -> index
    return stoi, itos

src_stoi, src_itos = build_vocab([e for e, _ in pairs])
tgt_stoi, tgt_itos = build_vocab([n for _, n in pairs])
print(f"Source (EN) vocab size: {len(src_itos)}")
print(f"Target (NE) vocab size: {len(tgt_itos)}")

def encode(sentence, stoi, add_bos_eos=True):
    ids = [stoi.get(t, UNK_IDX) for t in tokenize(sentence)]
    if add_bos_eos:
        ids = [BOS_IDX] + ids + [EOS_IDX]
    return ids

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.data = [(encode(e, src_stoi), encode(n, tgt_stoi)) for e, n in pairs]
    def __len__(self):  return len(self.data)
    def __getitem__(self, i):
        s, t = self.data[i]
        return torch.tensor(s), torch.tensor(t)

def collate(batch):
    src = pad_sequence([s for s, _ in batch], batch_first=True, padding_value=PAD_IDX)
    tgt = pad_sequence([t for _, t in batch], batch_first=True, padding_value=PAD_IDX)
    return src, tgt

loader = DataLoader(TranslationDataset(pairs), batch_size=8,
                    shuffle=True, collate_fn=collate)
xb, yb = next(iter(loader))
print("Batched source shape:", xb.shape, " target shape:", yb.shape)

### C.3 &mdash; The model

We reuse PyTorch's `nn.Transformer` for the attention stack and add our own **token embeddings**, **positional encoding** and output **projection**. The positional encoding is the classic sinusoidal formula from *Attention Is All You Need*.

In [ ]:
import math
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=100):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d_model)
    def forward(self, x):                              # x: (batch, seq, d_model)
        return self.dropout(x + self.pe[:, :x.size(1)])

class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=128, nhead=8,
                 num_encoder_layers=3, num_decoder_layers=3,
                 dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=PAD_IDX)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward, dropout=dropout,
            batch_first=True)
        self.generator = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt, tgt_mask, src_pad, tgt_pad):
        s = self.pos(self.src_emb(src) * math.sqrt(self.d_model))
        t = self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        out = self.transformer(s, t, tgt_mask=tgt_mask,
                               src_key_padding_mask=src_pad,
                               tgt_key_padding_mask=tgt_pad,
                               memory_key_padding_mask=src_pad)
        return self.generator(out)

    # helpers for inference
    def encode(self, src, src_pad):
        s = self.pos(self.src_emb(src) * math.sqrt(self.d_model))
        return self.transformer.encoder(s, src_key_padding_mask=src_pad)
    def decode(self, tgt, memory, tgt_mask, memory_pad):
        t = self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model))
        return self.transformer.decoder(t, memory, tgt_mask=tgt_mask,
                                        memory_key_padding_mask=memory_pad)

def subsequent_mask(sz):
    # float mask: 0 on/below diagonal, -inf above (blocks attention to the future)
    return torch.triu(torch.full((sz, sz), float("-inf")), diagonal=1)

nmt = Seq2SeqTransformer(len(src_itos), len(tgt_itos)).to(device)
n_params = sum(p.numel() for p in nmt.parameters())
print(f"Model built: {n_params:,} parameters")

### C.4 &mdash; Training loop

Standard teacher-forcing: feed the decoder the reference target shifted right, and predict the next token at every position. We ignore `<pad>` in the loss.

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(nmt.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)

EPOCHS = 80
nmt.train()
for epoch in range(1, EPOCHS + 1):
    total = 0.0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in  = tgt[:, :-1]          # decoder input  (drop last)
        tgt_out = tgt[:, 1:]           # expected output (drop first / <bos>)

        tgt_mask = subsequent_mask(tgt_in.size(1)).to(device)
        src_pad  = (src == PAD_IDX)
        tgt_pad  = (tgt_in == PAD_IDX)

        logits = nmt(src, tgt_in, tgt_mask, src_pad, tgt_pad)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(nmt.parameters(), 1.0)
        optimizer.step()
        total += loss.item()

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{EPOCHS} | loss {total/len(loader):.4f}")
print("Training complete.")

### C.5 &mdash; Greedy decoding (translate!)

At inference the target sentence does not exist yet, so we generate it token by token: start with `<bos>`, repeatedly pick the highest-probability next token, and stop at `<eos>`.

In [ ]:
@torch.no_grad()
def translate_scratch(sentence, max_len=20):
    nmt.eval()
    src = torch.tensor([encode(sentence, src_stoi)]).to(device)   # (1, src_len)
    src_pad = (src == PAD_IDX)
    memory = nmt.encode(src, src_pad)

    ys = torch.tensor([[BOS_IDX]]).to(device)
    for _ in range(max_len):
        tgt_mask = subsequent_mask(ys.size(1)).to(device)
        out = nmt.decode(ys, memory, tgt_mask, src_pad)
        logits = nmt.generator(out[:, -1])            # last position
        nxt = logits.argmax(-1).item()
        ys = torch.cat([ys, torch.tensor([[nxt]]).to(device)], dim=1)
        if nxt == EOS_IDX:
            break
    tokens = [tgt_itos[i] for i in ys[0].tolist()[1:]]  # drop <bos>
    if tokens and tokens[-1] == EOS:
        tokens = tokens[:-1]
    return " ".join(tokens)

# Sentences the model saw during training (should reproduce well)
for s in ["i love nepal", "the weather is nice", "where do you live",
          "please help me", "i am a student"]:
    print(f"EN: {s:28s} -> NE: {translate_scratch(s)}")

**What just happened?** Our hand-built Transformer learned the English&rarr;Nepali mapping from ~40 examples. Because the corpus is tiny it memorises rather than generalises &mdash; try a novel word combination and it will struggle. That is the expected lesson: *real* systems (Part A/B) are the same architecture trained on **millions** of sentence pairs.

**Scale it up yourself:** swap in a public parallel corpus such as [FLORES-200](https://github.com/facebookresearch/flores) or the Nepali portion of [OPUS](https://opus.nlpl.eu/), replace word-level tokenisation with a **SentencePiece/BPE** tokenizer (Nepali is morphologically rich), and increase model size and epochs.

## Part D &mdash; Interactive translation app (optional)

A small **Gradio** UI wrapping the pretrained NLLB model. Run the cell and a live widget appears inside Colab (and a temporary public link).

In [ ]:
import gradio as gr

def gradio_translate(text, direction):
    if not text.strip():
        return ""
    if direction == "English -> Nepali":
        return nllb_translate(text, "eng_Latn", "npi_Deva")[0]
    else:
        return nllb_translate(text, "npi_Deva", "eng_Latn")[0]

demo = gr.Interface(
    fn=gradio_translate,
    inputs=[gr.Textbox(lines=3, label="Input text"),
            gr.Radio(["English -> Nepali", "Nepali -> English"],
                     value="English -> Nepali", label="Direction")],
    outputs=gr.Textbox(lines=3, label="Translation"),
    title="Nepali NMT (NLLB-200)",
    description="Transformer-based English <-> Nepali translation | NCIT",
    examples=[["Nepal is a country of the Himalayas.", "English -> Nepali"],
              ["म प्रोग्रामिङ सिक्दैछु।", "Nepali -> English"]],
)
demo.launch(share=True, debug=False)

## Summary & exercises

**You built:**
- Instant EN&harr;NE translation with two pretrained Transformers (**NLLB-200**, **mBART-50**)
- A **BLEU** evaluation comparing them
- A **from-scratch** encoder&ndash;decoder Transformer trained on Nepali data
- An interactive **Gradio** app

**Exercises for students**
1. Add 20 new sentence pairs to the Part C corpus and retrain. Does BLEU on a held-out pair improve?
2. Replace word-level tokenisation with `sentencepiece` (BPE) and compare how it handles unseen Nepali words.
3. Fine-tune NLLB on a domain corpus (e.g. weather bulletins) using the `datasets` + `Trainer` API and re-measure BLEU.
4. Add **beam search** to `translate_scratch` and compare against greedy decoding.
5. Evaluate on the **FLORES-200** Nepali dev set instead of the toy test set.

---
*Prepared for AI / Machine Learning coursework &mdash; Nepal College of Information Technology (NCIT).*